In [8]:
pip install -U transformers torch pandas sentencepiece accelerate

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [9]:
import pandas as pd

df = pd.read_csv("Support_tickets.csv")

print(df.columns)

Index(['ticket_id', 'day_of_week', 'day_of_week_num', 'company_id',
       'company_size', 'company_size_cat', 'industry', 'industry_cat',
       'customer_tier', 'customer_tier_cat', 'org_users', 'region',
       'region_cat', 'past_30d_tickets', 'past_90d_incidents', 'product_area',
       'product_area_cat', 'booking_channel', 'booking_channel_cat',
       'reported_by_role', 'reported_by_role_cat', 'customers_affected',
       'error_rate_pct', 'downtime_min', 'payment_impact_flag',
       'security_incident_flag', 'data_loss_flag', 'has_runbook',
       'customer_sentiment', 'customer_sentiment_cat', 'description_length',
       'priority', 'priority_cat'],
      dtype='object')


In [10]:
text_columns = [col for col in df.columns if df[col].dtype == "object"]

print("Possible text columns:", text_columns)

Possible text columns: ['day_of_week', 'company_size', 'industry', 'customer_tier', 'region', 'product_area', 'booking_channel', 'reported_by_role', 'customer_sentiment', 'priority']


In [11]:
possible_cols = [col for col in df.columns if df[col].dtype == "object"]

# pick the column with longest average text length
text_col = max(possible_cols, key=lambda c: df[c].astype(str).str.len().mean())

print("Selected text column:", text_col)

Selected text column: product_area


In [12]:
ticket = df[text_col].iloc[0]
print(ticket)

mobile


In [13]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)

def create_prompt(ticket):
    return f"""
You are a support ticket classifier.

Ticket: {ticket}

Return only one label (Billing, Technical, Account, General).
"""

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM',

In [14]:
ticket = df[text_col].iloc[0]

prompt = create_prompt(ticket)

result = generator(prompt, max_new_tokens=10, do_sample=False)

print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=10) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a support ticket classifier.

Ticket: mobile

Return only one label (Billing, Technical, Account, General).



In [15]:
df["full_text"] = df[text_col].astype(str)

In [16]:
df["full_text"] = df[text_col].fillna("")

In [17]:
ticket = df["full_text"].iloc[0]